# Healthcare Prescription Analytics using Association Rule Learning

## Project Objective

The aim of this project is to look at prescription transactions and find medicines that tend to appear together.

We will work through the project in this order:

1. Understand the healthcare business problem
2. Import required Python libraries
3. Load and inspect the dataset
4. Identify and handle missing values
5. Standardize medicine names
6. Convert prescription dates into useful time features
7. Perform detailed Exploratory Data Analysis (EDA)
8. Transform prescription records into transaction baskets
9. Encode transactions into binary format
10. Apply the Apriori algorithm
11. Generate association rules
12. Evaluate rules using support, confidence, and lift
13. Interpret the strongest rules from a pharmacy and inventory perspective

> **Important:** Association rules identify statistical co-occurrence patterns. They must not be interpreted as medical treatment recommendations or proof of clinical appropriateness.

## Domain Knowledge

A hospital prescription may contain multiple medicines. In the raw dataset, each medicine appears as a separate row while the same `Prescription_ID` can repeat.

For example:

| Prescription_ID | Medicine_Name |
|---|---|
| RX001 | Paracetamol |
| RX001 | Vitamin C |
| RX001 | Azithromycin |
| RX002 | Metformin |
| RX002 | Telmisartan |

For Association Rule Learning, the repeated rows must be transformed into transaction baskets:

- **RX001** → Paracetamol, Vitamin C, Azithromycin
- **RX002** → Metformin, Telmisartan

This transaction structure allows us to identify medicines that frequently occur together.

## Association Rule Learning

Association Rule Learning is an **unsupervised learning technique** used to discover relationships between items that frequently appear together.

A rule is written as:

**Antecedent → Consequent**

Example:

**Medicine A → Medicine B**

This does not mean Medicine A causes Medicine B to be prescribed. It means that the two medicines occur together frequently enough to form a statistical association.

### Metrics we will use

#### Support
Support measures how frequently an itemset appears in all transactions.

**Support(A) = Number of transactions containing A / Total number of transactions**

A higher support value means the medicine or medicine combination appears frequently.

#### Confidence
Confidence measures how often the consequent appears when the antecedent is already present.

**Confidence(A → B) = Support(A ∪ B) / Support(A)**

A confidence of 0.80 means that 80% of transactions containing A also contain B.

#### Lift
Lift compares the observed association with what would be expected if A and B were independent.

**Lift(A → B) = Confidence(A → B) / Support(B)**

- **Lift > 1:** Positive association
- **Lift = 1:** No meaningful association
- **Lift < 1:** Negative association

Lift is particularly useful because a high-confidence rule can still be uninteresting if the consequent is already extremely common.

## About the Apriori Algorithm

Apriori is a classic frequent-itemset mining algorithm.

It works on the principle:

> If an itemset is frequent, all of its subsets must also be frequent.

### How Apriori works

1. Calculate the support of individual medicines.
2. Remove medicines that do not satisfy minimum support.
3. Generate candidate pairs from frequent medicines.
4. Calculate the support of those pairs.
5. Remove infrequent pairs.
6. Continue generating larger itemsets until no more frequent combinations remain.
7. Convert frequent itemsets into association rules.
8. Evaluate the rules using confidence, lift, and related metrics.

### Limitation

Apriori can become computationally expensive on large datasets because it repeatedly generates candidate itemsets. This is one reason FP-Growth is often used for larger transaction datasets.

## 1. Import Required Libraries

In [ ]:
# pandas for working with the dataset
import pandas as pd

# numpy for numerical operations
import numpy as np

# matplotlib for plots
import matplotlib.pyplot as plt

# seaborn for statistical plots
import seaborn as sns

# show all columns while inspecting the data
pd.set_option('display.max_columns', None)

# avoid cutting off longer text values
pd.set_option('display.max_colwidth', 100)


#observation: All the required libraries are imported successfully for data analysis and visualization. The settings are also changed to view the dataset properly.

## 2. Load the Healthcare Prescription Dataset

Keep the CSV file in the same folder as this notebook.

If your file is stored elsewhere, replace the value of `DATA_PATH` with the correct path.

In [ ]:
# Define the portable dataset path so the notebook can run on different computers.
DATA_PATH = "healthcare_prescription_analytics_dataset_no_data_quality_flag.csv"

# read the healthcare prescription CSV file into a pandas DataFrame.
df = pd.read_csv(DATA_PATH)

# display the first five records to verify that the dataset loaded successfully.
df.head()

#observation: The healthcare prescription dataset is loaded successfully into a DataFrame. The first five records are displayed to check the data.

## 3. Initial Dataset Inspection

In [ ]:
# Quickly display the first five records to understand the dataset structure.
df.head()

#observation: The first five rows of the dataset are displayed to get a basic idea about the data and its columns.

In [ ]:
# display the final five records to verify the dataset ending and formatting.
df.tail()

#observation: The last five rows of the dataset are displayed to check the ending records and data format.

In [ ]:
# inspect the number of rows and columns available in the dataset.
df.shape

#observation: The dataset contains a specific number of rows and columns, which shows the overall size of the dataset.

In [ ]:
# list every column name to understand the available analytical variables.
df.columns.tolist()

#observation: The column names are displayed, which helps us understand the different variables available in the dataset.

In [ ]:
# inspect data types, non-null counts, and memory usage.
df.info()

#observation: The info() output shows the data types and non-null values of each column. It helps us check the overall structure of the dataset.

In [ ]:
# Statistically summarize the numerical columns using count, mean, spread, and quartiles.
df.describe()

#observation: The describe() output gives the basic statistics of the numerical columns, such as mean, minimum, maximum, and quartiles.

In [ ]:
# summarize the categorical columns to inspect unique and dominant values.
df.describe(include='object')

#observation: The categorical columns are summarized with their unique values and most common values. This helps us understand the categorical data better.

### Interpretation

At this stage, check:

- Whether the expected columns are present
- Whether numerical columns have sensible minimum and maximum values
- Whether dates are stored as text
- Whether categorical columns contain inconsistent labels
- Whether null values exist
- Whether the dataset contains repeated prescription IDs, which is expected because one prescription can contain multiple medicines

## 4. Data Quality Checks

In [ ]:
# count missing values in every column to identify incomplete information.
missing_values = df.isnull().sum()

# display the missing-value count for each column.
missing_values

#obs: The missing values in each column are counted. This helps us identify which columns have incomplete data.

In [ ]:
# calculate the percentage of missing values in each column.
missing_percentage = (df.isnull().sum() / len(df)) * 100

# display missing-value percentages in descending order.
missing_percentage.sort_values(ascending=False)

#obs: The missing-value percentage for each column is calculated and sorted. This helps us identify the columns with the highest missing data.

In [ ]:
# count exact duplicate rows to identify redundant records.
duplicate_rows = df.duplicated().sum()

# display the total number of exact duplicate rows.
duplicate_rows

#obs: The duplicate rows are counted to check for repeated records in the dataset. This helps us identify any unnecessary duplicate data.

### Important Note on Duplicates

Do not remove rows simply because `Prescription_ID` repeats.

A repeated `Prescription_ID` is expected because one prescription may contain several different medicines.

Only **fully duplicated rows** should be considered true duplicates.

## 5. Handle Missing Diagnosis Values

In [ ]:
# inspect records where the diagnosis value is missing.
df[df['Diagnosis'].isnull()].head(10)

#obs: The records with missing diagnosis values are displayed. This helps us identify incomplete diagnosis information in the dataset.

In [ ]:
# replace missing diagnosis values with the meaningful label 'Unknown'.
df['Diagnosis'] = df['Diagnosis'].fillna('Unknown')

# verify that diagnosis values no longer contain null entries.
df['Diagnosis'].isnull().sum()

#obs: The missing diagnosis values are replaced with "Unknown". After checking, there are no missing values left in the Diagnosis column.

## 6. Standardize Medicine Names

In [ ]:
# count medicine-name occurrences to detect spelling inconsistencies.
medicine_name_counts = df['Medicine_Name'].value_counts()

# display all medicine frequencies for inspection.
medicine_name_counts

#obs: The frequency of each medicine name is displayed. This helps us identify common medicines and possible spelling inconsistencies.

In [ ]:
# standardize known medicine spelling variations into consistent names.
df['Medicine_Name'] = df['Medicine_Name'].replace({
    'Paracetmol': 'Paracetamol',
    'Vit C': 'Vitamin C',
    'Azithrocin': 'Azithromycin'
})

# recount medicine names to confirm successful standardization.
df['Medicine_Name'].value_counts()


## 7. Inspect Important Healthcare Categories

In [ ]:
# display all unique medicine categories available in the dataset.
df['Medicine_Category'].unique()
#The unique medicine categories in the dataset are displayed. This helps us understand the different types of medicines available.

In [ ]:
# display all unique diagnoses available after cleaning.
df['Diagnosis'].unique()

#obseravtion: The unique diagnosis values are displayed after cleaning. This helps us understand the different diagnoses present in the dataset.

In [ ]:
# display all unique hospital departments represented in the dataset.
df['Department'].unique()

#observation: The unique hospital departments are displayed. This helps us understand which departments are included in the dataset.

## 8. Date Preparation and Feature Engineering

In [ ]:
# convert the prescription-date column into a proper datetime format.
df['Prescription_Date'] = pd.to_datetime(df['Prescription_Date'])

# extract the calendar year for yearly healthcare trend analysis.
df['Year'] = df['Prescription_Date'].dt.year

# extract the numeric month for chronological grouping.
df['Month_Number'] = df['Prescription_Date'].dt.month

# extract the readable month name for intuitive visualizations.
df['Month_Name'] = df['Prescription_Date'].dt.month_name()

# extract the quarter for quarterly prescription analysis.
df['Quarter'] = df['Prescription_Date'].dt.quarter

# display the newly engineered date features.
df[['Prescription_Date', 'Year', 'Month_Number', 'Month_Name', 'Quarter']].head()

#obs: The prescription date is converted into datetime format, and new columns for year, month, and quarter are created.
#These columns will help in analyzing prescription trends over time.

# 9. Exploratory Data Analysis (EDA)

EDA helps us understand prescription behavior before building association rules.

The objective is to investigate:

- Most frequently prescribed medicines
- Department workload
- Common diagnoses
- Medicine-category distribution
- Patient-age distribution
- Gender distribution
- Monthly prescription trends
- City-wise prescription volume
- Medicine-cost patterns
- Department and medicine-category relationships

## 9.1 Top 10 Most Prescribed Medicines

In [ ]:

# count medicine occurrences and select the ten most frequently prescribed medicines.
top_medicines = df['Medicine_Name'].value_counts().head(10)

# display the top ten medicines and their frequencies.
top_medicines

#obs: The top 10 most frequently prescribed medicines are identified based on their prescription counts. This shows which medicines are prescribed the most.

In [ ]:
# set the figure size for a readable medicine-frequency chart.
plt.figure(figsize=(12, 6))

# plot the top ten medicines using an informative bar chart.
sns.barplot(x=top_medicines.index, y=top_medicines.values)

# rotate medicine labels to prevent overlapping text.
plt.xticks(rotation=45, ha='right')

# add a title to the visualization.
plt.title("Top 10 Prescribed Medicines")

# label the horizontal axis.
plt.xlabel("Medicine")

# label the vertical axis.
plt.ylabel("Prescription Row Count")

# optimize spacing around chart elements.
plt.tight_layout()

# show the plot
plt.show()

#The bar chart shows the top 10 prescribed medicines and their prescription counts. It helps us easily compare which medicines are prescribed more frequently.

### Interpretation

This chart identifies medicines with the highest prescription-row frequency.

From a pharmacy operations perspective, frequently prescribed medicines may require:

- Higher safety stock
- More frequent replenishment
- Closer stock-out monitoring

However, this chart measures individual medicine frequency and does **not** yet tell us which medicines are prescribed together.

## 9.2 Department-wise Prescription Volume

In [ ]:
# count prescription rows handled by each department.
department_counts = df['Department'].value_counts()

# display department-level prescription volume.
department_counts

#The number of prescriptions handled by each department is shown. This helps us identify which department has more prescription records.

In [ ]:
# set the figure size for department-level comparison.
plt.figure(figsize=(11, 6))

# plot departments in descending order of prescription-row frequency.
sns.countplot(data=df, x='Department', order=df['Department'].value_counts().index)

# rotate department labels for improved readability.
plt.xticks(rotation=45, ha='right')

# add a title to the departmental analysis.
plt.title("Department-wise Prescription Volume")

# label the horizontal axis.
plt.xlabel("Department")

# label the vertical axis.
plt.ylabel("Prescription Row Count")

# adjust spacing
plt.tight_layout()

# show the plot
plt.show()

#The bar chart compares prescription volume across different departments. It shows which departments have the highest and lowest number of prescription records.

## 9.3 Diagnosis Frequency

In [ ]:

# count the number of prescription rows associated with each diagnosis.
diagnosis_counts = df['Diagnosis'].value_counts()

# display diagnosis frequencies.
diagnosis_counts

#The number of prescriptions for each diagnosis is displayed. This helps us identify the most common and least common diagnoses in the dataset.

In [ ]:
# set the figure size to accommodate multiple diagnosis labels.
plt.figure(figsize=(14, 7))

# compare diagnosis frequencies in descending order.
sns.countplot(data=df, x='Diagnosis', order=df['Diagnosis'].value_counts().index)

# rotate diagnosis labels to make each category readable.
plt.xticks(rotation=90)

# add a title to the diagnosis-frequency visualization.
plt.title("Diagnosis Frequency")

# label the horizontal axis.
plt.xlabel("Diagnosis")

# label the vertical axis.
plt.ylabel("Prescription Row Count")

# adjust spacing
plt.tight_layout()

# show the plot
plt.show()

#The bar chart shows the frequency of different diagnoses. It helps us identify which diagnoses have more prescription records in the dataset.

### Interpretation

Diagnosis frequency helps explain **why** certain medicine groups may later appear together.

A highly represented diagnosis can produce frequent medicine combinations simply because it occurs often. Therefore, support and lift should be interpreted together when evaluating association rules.

## 9.4 Medicine Category Distribution

In [ ]:
# count the number of records belonging to each medicine category.
medicine_category_counts = df['Medicine_Category'].value_counts()

# display medicine-category frequencies.
medicine_category_counts
#The number of prescriptions in each medicine category is displayed. This helps us see which medicine categories are more common in the dataset.


In [ ]:
# set the figure size for categorical medicine analysis.
plt.figure(figsize=(12, 7))

# display medicine-category frequencies using a readable horizontal bar chart.
sns.barplot(x=medicine_category_counts.values, y=medicine_category_counts.index)

# add a title to the medicine-category analysis.
plt.title("Medicine Category Distribution")

# label the horizontal axis.
plt.xlabel("Prescription Row Count")

# label the vertical axis.
plt.ylabel("Medicine Category")

# adjust spacing
plt.tight_layout()

# show the plot
plt.show()

#Observation:
#The bar chart shows the distribution of different medicine categories. It helps us compare which categories have more and fewer prescription records.

## 9.5 Patient Age Distribution

In [ ]:
# set the figure size for examining patient-age distribution.
plt.figure(figsize=(10, 5))

# plot patient ages using a detailed histogram with a smooth density curve.
sns.histplot(data=df, x='Patient_Age', bins=30, kde=True)

# add a title to the patient-age analysis.
plt.title("Patient Age Distribution")

# label the horizontal axis.
plt.xlabel("Patient Age")

# label the vertical axis.
plt.ylabel("Prescription Row Count")

# adjust spacing
plt.tight_layout()

# show the plot
plt.show()

#Observation:
#The histogram shows the distribution of patient ages in the dataset. It helps us understand the common age groups of the patients.



## 9.6 Gender Distribution

In [ ]:
# set the figure size for gender-distribution analysis.
plt.figure(figsize=(7, 5))

# compare prescription-row counts across gender categories.
sns.countplot(data=df, x='Gender', order=df['Gender'].value_counts().index)

# add a title to the gender-distribution chart.
plt.title("Gender Distribution")

# label the horizontal axis.
plt.xlabel("Gender")

# label the vertical axis.
plt.ylabel("Prescription Row Count")

# adjust spacing
plt.tight_layout()

# show the plot
plt.show()

#Observation:The count plot shows the number of prescription records for each gender. It helps us compare the distribution of patients by gender.


## 9.7 Monthly Prescription Trend

In [ ]:
# set the figure size for the monthly prescription trend.
plt.figure(figsize=(12, 5))

# plot month-wise prescription-row counts using a clear line chart.
plt.plot(monthly_counts.index, monthly_counts.values, marker='o')

# add a title to the monthly trend analysis.
plt.title("Monthly Prescription Trend")

# label the horizontal axis.
plt.xlabel("Month")

# label the vertical axis.
plt.ylabel("Prescription Row Count")

# rotate month labels for comfortable reading.
plt.xticks(rotation=45)

# adjust spacing
plt.tight_layout()

# show the plot
plt.show()

#Observation:The line chart shows how the number of prescriptions changes from month to month. It helps us identify months with higher and lower prescription counts.

## 9.8 Top Cities by Prescription Volume

In [ ]:
# identify the ten cities with the highest prescription-row counts.
city_counts = df['City'].value_counts().head(10)

# display the leading cities and their prescription volumes.
city_counts

#Observation:The top 10 cities with the highest number of prescription records are identified. This helps us find the cities with more prescription activity.


In [ ]:
# set the figure size for city-level comparison.
plt.figure(figsize=(12, 5))

# plot the ten cities with the highest prescription-row volumes.
sns.barplot(x=city_counts.index, y=city_counts.values)

# rotate city labels for improved readability.
plt.xticks(rotation=45, ha='right')

# add a title to the city-volume analysis.
plt.title("Top Cities by Prescription Volume")

# label the horizontal axis.
plt.xlabel("City")

# label the vertical axis.
plt.ylabel("Prescription Row Count")

# adjust spacing
plt.tight_layout()

# show the plot
plt.show()

#Observation:The bar chart shows the top 10 cities based on prescription volume. It helps us compare which cities have more prescription records.


## 9.9 Medicine Cost Analysis

In [ ]:
# calculate the average unit price across all medicine records.
average_unit_price = df['Unit_Price'].mean()

# display the average unit price rounded to two decimal places.
round(average_unit_price, 2)

#Observation:The average unit price of the medicines is calculated and rounded to two decimal places. This gives an idea of the typical price of medicines in the dataset.


In [ ]:
# calculate the average unit price for each standardized medicine.
average_price_by_medicine = df.groupby('Medicine_Name')['Unit_Price'].mean()

# sort medicines from the highest to the lowest average unit price.
most_expensive_medicines = average_price_by_medicine.sort_values(ascending=False).head(10)

# display the ten medicines with the highest average unit prices.
most_expensive_medicines

#Observation:The average price of each medicine is calculated and the top 10 most expensive medicines are identified. This helps us compare medicines based on their average unit price.


In [ ]:
# set the figure size for medicine-cost comparison.
plt.figure(figsize=(12, 6))

# compare the ten medicines with the highest average unit prices.
sns.barplot(x=most_expensive_medicines.values, y=most_expensive_medicines.index)

# add a title to the medicine-cost visualization.
plt.title("Top 10 Medicines by Average Unit Price")

# label the horizontal axis.
plt.xlabel("Average Unit Price")

# label the vertical axis.
plt.ylabel("Medicine")

# adjust spacing
plt.tight_layout()

# show the plot
plt.show()

#Observation:The bar chart shows the 10 medicines with the highest average unit prices. It helps us compare the prices and identify the most expensive medicines.

## 9.10 Department vs Medicine Category

In [ ]:
# create a cross-tabulation between departments and medicine categories.
department_category_table = pd.crosstab(df['Department'], df['Medicine_Category'])

# display the department-by-category frequency matrix.
department_category_table


In [ ]:
# set the figure size for the department-category relationship.
plt.figure(figsize=(14, 7))

# display category usage across departments using an annotated heatmap.
sns.heatmap(department_category_table, cmap='YlGnBu')

# add a title to the department-category heatmap.
plt.title("Department vs Medicine Category")

# label the horizontal axis.
plt.xlabel("Medicine Category")

# label the vertical axis.
plt.ylabel("Department")

# adjust spacing
plt.tight_layout()

# show the plot
plt.show()


# 10. Prepare Transactions for Association Rule Learning

Apriori cannot directly use the original row-level DataFrame.

We first need to transform prescription rows into **medicine baskets** where each prescription represents one transaction.

The transformation occurs in three stages:

1. Group medicine names by `Prescription_ID`
2. Create a prescription × medicine matrix
3. Convert counts into Boolean presence/absence values

## 10.1 Create Prescription Transactions

In [ ]:
# group medicine names by prescription so each prescription becomes one transaction.
transactions = df.groupby('Prescription_ID')['Medicine_Name'].apply(list)

# display the first five medicine transactions.
transactions.head()


### Interpretation

Each row now represents one prescription basket.

Example:

`RX000001 → [Pantoprazole, Ondansetron, ...]`

This is much closer to the structure required for Association Rule Learning.

## 10.2 Build the Medicine Basket Matrix

In [ ]:
# count each medicine within every prescription-level transaction.
basket = df.groupby(['Prescription_ID', 'Medicine_Name'])['Medicine_Name'].count().unstack(fill_value=0)

# display the first five rows of the transaction basket.
basket.head()


In the basket matrix:

- Rows = Prescription IDs
- Columns = Medicines
- Values = Number of times a medicine appears in that prescription

Apriori only needs to know whether a medicine is present, so the next step converts the counts into Boolean values.

In [ ]:
# convert medicine counts into Boolean presence values required by modern mlxtend.
basket = basket.astype(bool)

# display the Boolean basket matrix for verification.
basket.head()


In [ ]:
# inspect the number of prescription transactions and unique medicine columns.
basket.shape


In [ ]:
# count how many prescription baskets contain each medicine.
medicine_frequency = basket.sum().sort_values(ascending=False)

# display the ten most frequent medicines at transaction level.
medicine_frequency.head(10)


### Why Transaction Frequency Can Differ from Earlier EDA

Earlier EDA counted **medicine rows**.

The basket frequency counts the number of **unique prescriptions containing each medicine**.

For Association Rule Learning, transaction-level frequency is the correct measure.

In [ ]:
# set the figure size for transaction-level medicine-frequency analysis.
plt.figure(figsize=(12, 6))

# plot the ten medicines appearing in the highest number of prescription baskets.
medicine_frequency.head(10).plot(kind='bar')

# add a title to the transaction-level frequency chart.
plt.title("Top 10 Most Frequent Medicines by Prescription Basket")

# label the horizontal axis.
plt.xlabel("Medicine")

# label the vertical axis.
plt.ylabel("Number of Prescriptions")

# rotate medicine labels for improved readability.
plt.xticks(rotation=45, ha='right')

# adjust spacing
plt.tight_layout()

# show the plot
plt.show()


# 11. Install and Import Apriori Tools

The project uses the `mlxtend` library.

If `mlxtend` is already installed, the installation cell can be skipped.

In [ ]:
# install the mlxtend package required for Association Rule Learning.
%pip install -q mlxtend


In [ ]:
# Import the classic Apriori function for frequent-itemset discovery.
from mlxtend.frequent_patterns import apriori

# Import the analytical association-rules function for rule generation.
from mlxtend.frequent_patterns import association_rules


# 12. Apply the Apriori Algorithm

We will begin with:

`min_support = 0.02`

This means an itemset must occur in at least **2% of all prescription transactions** to be considered frequent.

The threshold should not be chosen blindly. In a real project, experiment with different support values and evaluate:

- Number of frequent itemsets
- Number of generated rules
- Business usefulness
- Computational cost

In [ ]:
# generate frequent medicine itemsets using a meaningful minimum support threshold.
frequent_itemsets = apriori(basket, min_support=0.02, use_colnames=True)

# display the first few frequent medicine itemsets.
frequent_itemsets.head()


In [ ]:
# calculate the number of medicines contained in every frequent itemset.
frequent_itemsets['itemset_length'] = frequent_itemsets['itemsets'].apply(len)

# sort frequent itemsets from highest to lowest support.
frequent_itemsets_sorted = frequent_itemsets.sort_values(by='support', ascending=False)

# display the strongest frequent itemsets.
frequent_itemsets_sorted.head(20)


### Interpreting Frequent Itemsets

A frequent itemset such as:

`{Medicine A, Medicine B}`

with support `0.08` means that **8% of all prescription transactions contain both medicines**.

Frequent itemsets are not yet directional rules. They simply identify combinations that occur often enough to satisfy the support threshold.

# 13. Generate Association Rules

In [ ]:
# generate directional medicine rules using lift as the initial screening metric.
rules = association_rules(frequent_itemsets, metric='lift', min_threshold=1.0)

# display the first few generated association rules.
rules.head()


The generated rule table contains several useful metrics, including:

- `antecedents`
- `consequents`
- `antecedent support`
- `consequent support`
- `support`
- `confidence`
- `lift`
- `leverage`
- `conviction`

For this project, the main teaching focus is on **support, confidence, and lift**.

In [ ]:
# select the most important columns for readable rule interpretation.
rules_view = rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].copy()

# sort rules from the strongest to the weakest lift value.
rules_view = rules_view.sort_values(by='lift', ascending=False)

# display the twenty highest-lift association rules.
rules_view.head(20)


# 14. Filter Strong Association Rules

In [ ]:
# filter rules that show both meaningful confidence and strong positive lift.
strong_rules = rules[(rules['confidence'] >= 0.50) & (rules['lift'] > 1.20)].copy()

# sort the filtered rules from highest to lowest lift.
strong_rules = strong_rules.sort_values(by=['lift', 'confidence'], ascending=False)

# display the key metrics for the strongest filtered rules.
strong_rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(30)


## How to Interpret a Rule

Suppose a rule is:

**{Medicine A} → {Medicine B}**

with:

- Support = 0.07
- Confidence = 0.65
- Lift = 2.10

Interpretation:

- **Support 0.07:** 7% of all prescription transactions contain both medicines.
- **Confidence 0.65:** 65% of prescriptions containing Medicine A also contain Medicine B.
- **Lift 2.10:** Medicine B occurs with Medicine A about 2.1 times as often as expected under independence.

### Healthcare Caution

This statistical relationship can support:

- Pharmacy inventory planning
- Co-demand analysis
- Stock placement decisions
- Prescription-pattern research

It should **not** be presented as evidence that the medicines should clinically be prescribed together.

# 15. Make Rule Output Easier to Read

In [ ]:
# define a helper function that converts frozenset values into readable comma-separated text.
def readable_itemset(itemset):
    # join the sorted item names into one readable string.
    return ', '.join(sorted(list(itemset)))

# copy the filtered rules so the original analytical results remain unchanged.
readable_rules = strong_rules.copy()

# convert antecedent frozensets into readable medicine names.
readable_rules['antecedents'] = readable_rules['antecedents'].apply(readable_itemset)

# convert consequent frozensets into readable medicine names.
readable_rules['consequents'] = readable_rules['consequents'].apply(readable_itemset)

# display the final readable rule table.
readable_rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(30)


# 16. Visualize Strong Apriori Rules

In [ ]:
# select the fifteen highest-lift rules for focused visualization.
top_apriori_rules = readable_rules.head(15).copy()

# create a descriptive rule label combining antecedent and consequent medicines.
top_apriori_rules['Rule'] = top_apriori_rules['antecedents'] + ' → ' + top_apriori_rules['consequents']

# set the figure size for comparing rule strength.
plt.figure(figsize=(12, 8))

# compare the selected association rules by lift value.
sns.barplot(data=top_apriori_rules, x='lift', y='Rule')

# add a title to the strongest Apriori-rule visualization.
plt.title("Top Apriori Association Rules by Lift")

# label the horizontal axis.
plt.xlabel("Lift")

# label the vertical axis.
plt.ylabel("Association Rule")

# adjust spacing
plt.tight_layout()

# show the plot
plt.show()


In [ ]:
# set the figure size for examining rule-quality relationships.
plt.figure(figsize=(10, 6))

# plot support against confidence while representing lift through bubble size.
sns.scatterplot(data=rules, x='support', y='confidence', size='lift', sizes=(30, 300), alpha=0.7)

# add a title to the rule-quality scatter plot.
plt.title("Apriori Rules: Support vs Confidence")

# label the horizontal axis.
plt.xlabel("Support")

# label the vertical axis.
plt.ylabel("Confidence")

# adjust spacing
plt.tight_layout()

# show the plot
plt.show()


# 17. Interpreting the Results

For each strong rule, students should answer:

1. **What does the rule say?**
2. **How common is the combination?** → Support
3. **How reliable is the directional relationship?** → Confidence
4. **Is the relationship stronger than random co-occurrence?** → Lift
5. **What pharmacy or operational decision could the pattern support?**
6. **Could diagnosis, department, season, or patient profile explain the rule?**
7. **Does the rule require clinical validation before any healthcare use?** → Always yes

### Example Business Uses

Strong co-prescription patterns may support:

- Coordinated stock replenishment
- Detection of medicines with linked demand
- Department-specific stock allocation
- Seasonal pharmacy inventory planning
- Identification of prescription patterns for further clinical review

# 18. What to Include in the Conclusion

The conclusion should go beyond simply saying that Apriori generated association rules.

A stronger conclusion should discuss:

- Which medicines appeared most frequently
- Which departments and diagnoses generated the highest prescription activity
- Whether prescription demand showed seasonal patterns
- How the raw dataset was converted into transaction baskets
- How the minimum-support threshold affected frequent itemsets
- Which rules had the strongest confidence and lift
- How those rules could support pharmacy inventory decisions
- Why association does not imply clinical causation
- Limitations of Apriori on large transaction datasets

## Next Stage

After completing this notebook, the same basket can be analyzed using **FP-Growth** and the results can be compared with Apriori for:

- Execution speed
- Frequent itemsets
- Number of rules
- Rule quality
- Scalability